In [1]:
import pandas as pd
import splink.comparison_library as cl
from splink import Linker, SettingsCreator, block_on, DuckDBAPI

# Load data
data1 = pd.read_csv("csv_outputs/deduped_data_owner_president_ceo_founder.csv", dtype="string")
data1["record_id"] = data1["cluster_id"]
data2 = pd.read_csv("csv_outputs/additional_states/clean_df_owner_president_ceo_founder_2025-12-24.csv", dtype='string')
# df = data[data["company_state"] == "Oklahoma"].copy()
df = pd.concat([data1, data2]).reset_index(drop=True)
# df = df.drop(columns=['home_county', 'company_county'])
df["company_employee_size_actual"] = pd.to_numeric(df["company_employee_size_actual"], errors="coerce")
print(f"Total rows: {len(df):,}")

# Add unique ID required by splink - already crreated with record_id

# Clean key columns for better matching
df["first_name_clean"] = df["first_name"].str.replace(r'[^a-zA-Z]', '', regex=True).str.strip().str.lower()
df["last_name_clean"] = df["last_name"].str.replace(r'[^a-zA-Z]', '', regex=True).str.strip().str.lower()
df["forename_surname_concat_col_name"] = df["first_name_clean"] + " " + df["last_name_clean"]
df["company_name_clean"] = df["company_name"].str.replace(r'[^a-zA-Z0-9]', '', regex=True).str.strip().str.lower()
df["email_clean"] = df["email"].str.replace(" ", "").str.lower().str.strip()
df['linkedin_clean'] = df['linkedin'].str.split("/in/").str[-1].str.lower()
df = df.drop(columns=["cluster_id"])

Total rows: 1,070,803


In [2]:
# Define splink settings
# Blocking rules reduce the number of comparisons dramatically
# These rules determine which record pairs to compare

settings = SettingsCreator(
    link_type="dedupe_only",
    unique_id_column_name="record_id",
    
    # Blocking rules - records must match on at least one of these to be compared 
    blocking_rules_to_generate_predictions=[
        block_on("first_name_clean", "last_name_clean"),
        block_on("first_name_clean", "company_name_clean"),
        block_on("last_name_clean", "company_name_clean", "email_clean"),
        block_on("linkedin_clean", "company_name_clean"),
    ],
    comparisons=[
        cl.ForenameSurnameComparison("first_name_clean", "last_name_clean", forename_surname_concat_col_name="forename_surname_concat_col_name"),
        cl.ExactMatch("email_clean"),
        cl.ExactMatch("linkedin_clean"),
        cl.NameComparison("company_name_clean"),
        cl.ExactMatch("company_zipcode"),
    ],
    
    # Retain these columns in results
    retain_intermediate_calculation_columns=True,
)
# Initialize the linker with DuckDB backend (fast, in-memory)
linker = Linker(df, settings, db_api=DuckDBAPI())


In [3]:

# Estimate probability two random records match (u probability)
# This uses random sampling, so it's fast
linker.training.estimate_u_using_random_sampling(max_pairs=int(1e8))

# Estimate m probabilities using Expectation Maximization
# This learns how likely matching records are to agree on each field

linker.training.estimate_parameters_using_expectation_maximisation(
    block_on("first_name_clean", "last_name_clean")
)

linker.training.estimate_parameters_using_expectation_maximisation(
    block_on("first_name_clean", "company_name_clean")
)

linker.training.estimate_parameters_using_expectation_maximisation(
    block_on("email_clean") 
)

linker.training.estimate_parameters_using_expectation_maximisation(
    block_on("linkedin_clean")
)

linker.training.estimate_parameters_using_expectation_maximisation(
    block_on("company_zipcode", "last_name_clean")  
)

----- Estimating u probabilities using random sampling -----
u probability not trained for email_clean - Exact match on email_clean (comparison vector value: 1). This usually means the comparison level was never observed in the training data.

Estimated u probabilities using random sampling

Your model is not yet fully trained. Missing estimates for:
    - first_name_clean_last_name_clean (no m values are trained).
    - email_clean (some u values are not trained, no m values are trained).
    - linkedin_clean (no m values are trained).
    - company_name_clean (no m values are trained).
    - company_zipcode (no m values are trained).

----- Starting EM training session -----

Estimating the m probabilities of the model by blocking on:
(l."first_name_clean" = r."first_name_clean") AND (l."last_name_clean" = r."last_name_clean")

Parameter estimates will be made for the following comparison(s):
    - email_clean
    - linkedin_clean
    - company_name_clean
    - company_zipcode

Param

<EMTrainingSession, blocking on (l."company_zipcode" = r."company_zipcode") AND (l."last_name_clean" = r."last_name_clean"), deactivating comparisons first_name_clean_last_name_clean, company_zipcode>

In [4]:

linker.visualisations.match_weights_chart()


c:\Projects\Freelanxur\upwork_42168194-data_merge\.venv-upwork_42168194-data_merge\Lib\site-packages\altair\vegalite\v6\api.py:4124: UserWarning: Automatically deduplicated selection parameter with identical configuration. If you want independent parameters, explicitly name them differently (e.g., name='param1', name='param2'). See https://github.com/vega/altair/issues/3891
  return _tp.from_dict(dct, validate=validate)


alt.VConcatChart(...)

In [5]:
# Predict matches - this is where the actual deduplication happens
# threshold controls minimum match probability (0.0 to 1.0)
# Lower = more matches (but more false positives)
# Higher = fewer matches (but may miss some true duplicates)

print("Running predictions... (this may take a few minutes for 1.4M rows)")
predictions = linker.inference.predict(threshold_match_probability=0.95) # this is just a filter to store less data
pairwise_predictions = predictions.as_pandas_dataframe()
print("Predictions complete!")


Running predictions... (this may take a few minutes for 1.4M rows)


Blocking time: 6.46 seconds
Predict time: 6.87 seconds

 -- WARNING --
You have called predict(), but there are some parameter estimates which have neither been estimated or specified in your settings dictionary.  To produce predictions the following untrained trained parameters will use default values.
Comparison: 'email_clean':
    u values not fully trained
The 'probability_two_random_records_match' setting has been set to the default value (0.0001). 
If this is not the desired behaviour, either: 
 - assign a value for `probability_two_random_records_match` in your settings dictionary, or 
 - estimate with the `linker.training.estimate_probability_two_random_records_match` function.


Predictions complete!


In [6]:

# Cluster the predictions into groups of duplicates
# Each cluster represents a unique entity
clusters = linker.clustering.cluster_pairwise_predictions_at_threshold(
    predictions, 
    threshold_match_probability=0.98 # this is probably that connects records
)

df_with_clusters = clusters.as_pandas_dataframe().astype("string")
print(f"Number of clusters: {df_with_clusters['cluster_id'].nunique():,}")
print(f"Records in clusters: {len(df_with_clusters):,}")
df_with_clusters

Completed iteration 1, num edges remaining to process: 4374
Completed iteration 2, num edges remaining to process: 116
Completed iteration 3, num edges remaining to process: 52
Completed iteration 4, num edges remaining to process: 12
Completed iteration 5, num edges remaining to process: 4
Completed iteration 6, num edges remaining to process: 0


Number of clusters: 1,042,771
Records in clusters: 1,070,803


,cluster_id,source_file,record_id,first_name,last_name,job_title,email,work_phone,company_linkedin,facebook,...,general_2020,primary_2020,home_zipfour,forename_surname_concat_col_name,company_country,first_name_clean,last_name_clean,company_name_clean,email_clean,linkedin_clean
0,624029,['Zoominfo scraping 3 - Virginia.csv'],624029,Page,Allen,Owner,pallen@pageallenlaw.com,+1 804-385-7699,<NA>,<NA>,...,<NA>,<NA>,<NA>,page allen,<NA>,page,allen,pageallenlaw,pallen@pageallenlaw.com,page-allen-4855776
1,624031,['Zoominfo scraping 3 - Virginia.csv'],624031,Vicki,Francois,Founder & Attorney,vwiese@wieselawfirm.com,+1 540-815-9197,<NA>,<NA>,...,<NA>,<NA>,<NA>,vicki francois,<NA>,vicki,francois,wieselawfirmllc,vwiese@wieselawfirm.com,vicki-wiese
2,624038,['Zoominfo scraping 3 - Virginia.csv'],624038,Bobbi,Heaney,Business Owner,bheaney@mangosalon.com,+1 804-314-9561,<NA>,https://www.facebook.com/mangosalonrva,...,<NA>,<NA>,<NA>,bobbi heaney,<NA>,bobbi,heaney,mangosalon,bheaney@mangosalon.com,bobbi-bobbi-heaney-b0237665
3,624042,['Zoominfo scraping 3 - Virginia.csv'],624042,Harmony,Stearns,Small Business Owner,harmony@momsinmotion.net,+1 540-671-1966,<NA>,https://facebook.com/momsinmotionva,...,<NA>,<NA>,<NA>,harmony stearns,<NA>,harmony,stearns,momsinmotion,harmony@momsinmotion.net,harmony-stearns-384a7a69
4,624043,['Zoominfo scraping 3 - Virginia.csv'],624043,Tom,Farris,Owner,tom.farris@farriswater.com,+1 319-480-8882,<NA>,<NA>,...,<NA>,<NA>,<NA>,tom farris,<NA>,tom,farris,farrisenterprises,tom.farris@farriswater.com,tom-farris-77a316102
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1070798,624018,['Zoominfo scraping 3 - Virginia.csv'],624018,Joshua,Snyder,Business Owner,joshua_snyder@imsolutionsva.com,+1 610-223-2333,<NA>,https://facebook.com/industrialmaintenancesolu...,...,<NA>,<NA>,<NA>,joshua snyder,<NA>,joshua,snyder,industrialmaintenancesolutions,joshua_snyder@imsolutionsva.com,industrialmaintenancesolutions
1070799,624019,['Zoominfo scraping 3 - Virginia.csv'],624019,Bill,Lopez,President And Cio,blopez@ssva.com,+1 516-250-2501,<NA>,<NA>,...,<NA>,<NA>,<NA>,bill lopez,<NA>,bill,lopez,ssva,blopez@ssva.com,bill-lopez-4707304
1070800,624022,['Zoominfo scraping 3 - Virginia.csv'],624022,Cmtpt,Roberts,Business Owner And Pt And Mt,scott@robertspt.com,+1 804-840-3157,<NA>,<NA>,...,<NA>,<NA>,<NA>,cmtpt roberts,<NA>,cmtpt,roberts,robertsphysicaltherapymassage,scott@robertspt.com,scott-roberts-pt-mpt-comt-cmtpt-mcmt-fsnp-cmt-...
1070801,624024,['Zoominfo scraping 3 - Virginia.csv'],624024,Mark,Black,President,blackm@fuma.org,+1 434-924-0972,<NA>,https://facebook.com/forkunionmilitary,...,<NA>,<NA>,<NA>,mark black,<NA>,mark,black,forkunionmilitaryacademy,blackm@fuma.org,markedwardblack


In [7]:
df_with_clusters[df_with_clusters.duplicated(subset=["cluster_id"], keep=False)].sort_values(by="cluster_id")

,cluster_id,source_file,record_id,first_name,last_name,job_title,email,work_phone,company_linkedin,facebook,...,general_2020,primary_2020,home_zipfour,forename_surname_concat_col_name,company_country,first_name_clean,last_name_clean,company_name_clean,email_clean,linkedin_clean
851398,1000029,['[STATE] raw_data_states\\slack\\florida 904 ...,1001799,Heather,Mullis,Owner,<NA>,(904) 647-5466,<NA>,<NA>,...,<NA>,<NA>,<NA>,heather mullis,<NA>,heather,mullis,flamingoplumber,<NA>,<NA>
849569,1000029,['[STATE] raw_data_states\\slack\\florida 904 ...,1000029,Heather,Mullis,Owner,heather@flamingoseptictanks.com,(904) 940-4884,http://www.linkedin.com/company/flamingo-septi...,http://www.facebook.com//flamingoseptic,...,<NA>,<NA>,'4153,heather mullis,<NA>,heather,mullis,flamingosepticpumping,heather@flamingoseptictanks.com,heather-mullis-a015322a
849652,1000105,['[STATE] raw_data_states\\slack\\florida 904 ...,1000105,Norberto,Sanchez,President,<NA>,(904) 739-3660,<NA>,<NA>,...,<NA>,<NA>,<NA>,norberto sanchez,<NA>,norberto,sanchez,wnnrradio,<NA>,<NA>
192404,1000105,"['Rob Master Sheet (Updated).csv', '_Rob Maste...",247682,Norberto,Sanchez,Chief Executive Officer,norsan@norsangroup.com,+1 704-877-7138,<NA>,https://facebook.com/pages/norsan-group/246651...,...,<NA>,<NA>,<NA>,norberto sanchez,<NA>,norberto,sanchez,norsan,norsan@norsangroup.com,norberto-sanchez-ab44a032
598793,1000105,[STATE] raw_data_states/2025-12-20\georgia.xlsx,ASF30524,Norberto,Sanchez,Owner,<NA>,(770) 414-6532,<NA>,http://www.facebook.com/pages/norsan-group/246...,...,<NA>,<NA>,<NA>,norberto sanchez,<NA>,norberto,sanchez,norsandistributor,<NA>,<NA>
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
639040,ASF99927,[STATE] raw_data_states/2025-12-20\minnesota.xlsx,ASF99929,Roger,Sit,Chief Executive Officer,<NA>,(612) 332-3223,<NA>,<NA>,...,<NA>,<NA>,<NA>,roger sit,<NA>,roger,sit,sitlargecapgrowthfundinc,<NA>,<NA>
639041,ASF99927,[STATE] raw_data_states/2025-12-20\minnesota.xlsx,ASF99930,Roger,Sit,Chief Executive Officer,<NA>,(612) 359-2540,<NA>,<NA>,...,<NA>,<NA>,<NA>,roger sit,<NA>,roger,sit,sitmutualfunds,<NA>,<NA>
90733,ASF99952,[STATE] raw_data_states/2025-12-20\minnesota.xlsx,ASF99954,Corey,Johnson,President,<NA>,(612) 677-2500,http://www.linkedin.com/company/solve-branding...,https://www.facebook.com/solveagency,...,<NA>,<NA>,<NA>,corey johnson,<NA>,corey,johnson,solveadvertisingbranding,<NA>,<NA>
90732,ASF99952,[STATE] raw_data_states/2025-12-20\minnesota.xlsx,ASF99953,Corey,Johnson,President,<NA>,(612) 677-2500,<NA>,https://www.facebook.com/solveagency,...,<NA>,<NA>,<NA>,corey johnson,<NA>,corey,johnson,solveadvertisingbranding,<NA>,<NA>


In [8]:
# Define priority for source files 1 = best, 2 = middle, 3 = worst
df_with_clusters["source_file_sort_priority"] = 2  # default
df_with_clusters.loc[df_with_clusters["source_file"].str.contains("[STATE]", regex=False, na=False), "source_file_sort_priority"] = 1
df_with_clusters.loc[df_with_clusters["source_file"].isin(["Combined Master File 2_1", "www.csv"]), "source_file_sort_priority"] = 3

df_with_clusters = df_with_clusters.sort_values(by="source_file_sort_priority", ascending=True)
df_with_clusters = df_with_clusters.drop(columns=["source_file_sort_priority"] + [x for x in df_with_clusters.columns.tolist() if x.endswith("_clean")])

# Define aggregation rules based on column name patterns
# Format: (condition_function, aggregation_function)
# The condition_function takes a column name and returns True if the rule applies
# The aggregation_function is the pandas agg function to use

def agg_join(series):
    return series.tolist()

def agg_longest(series):
    """Return the longest non-null string value"""
    non_null = series.dropna()
    if len(non_null) == 0:
        return None
    return max(non_null, key=len)

def agg_phone_numbers(series):
    non_null = series.dropna()
    if len(non_null) == 0:
        return None
    
    # Priority 1: Has ( or )
    mask = non_null.str.contains(r'[()]', regex=True, na=False)
    if mask.any():
        return non_null[mask].iloc[0]
    
    # Priority 2: Has +
    mask = non_null.str.contains(r'\+', regex=True, na=False)
    if mask.any():
        return non_null[mask].iloc[0]
    
    # Priority 3: Longest value
    if len(non_null) > 0:
        longest_val = max(non_null, key=len)
        return longest_val
    
    return non_null.iloc[0]

def agg_mode(series):
    """Return the most frequently occurring non-null value"""
    non_null = series.dropna()
    if len(non_null) == 0:
        return None
    mode_result = non_null.mode()
    return mode_result.iloc[0] if len(mode_result) > 0 else non_null.iloc[0]

# ============================================================
# CUSTOMIZE YOUR AGGREGATION RULES HERE
# ============================================================
# Each rule is a tuple: (condition, aggregation_name/function)
# Conditions are checked in order - first match wins
# Available built-in agg names: 'first', 'last', 'min', 'max', 'sum', 'mean', 'count'
# Or use custom functions defined above

aggregation_rules = [
    # Example: Join all unique_ids together
    (lambda col: col == "source_file", agg_join),
    (lambda col: col == 'record_id', agg_join),
    (lambda col: 'work_phone' in col.lower(), agg_phone_numbers),
    (lambda col: 'mobile_phone' in col.lower(), agg_phone_numbers),
    (lambda col: 'landline_phone' in col.lower(), agg_phone_numbers),
    (lambda col: 'description' in col.lower(), agg_longest),
    # (lambda col: 'address' in col.lower(), agg_longest),
    (lambda col: 'actual' in col.lower(), "max"),
    (lambda col: 'volume' in col.lower(), "max"),
    (lambda col: 'revenue' in col.lower(), "max"),
    (lambda col: col == 'age', "max"),
]

# Default aggregation for columns that don't match any rule
default_agg = 'first'

# Build the aggregation dictionary for each column
def get_agg_function(col_name):
    """Return the appropriate aggregation function for a column"""
    for condition, agg_func in aggregation_rules:
        if condition(col_name):
            return agg_func
    return default_agg

# Get columns to aggregate (exclude group columns)
agg_cols = [col for col in df_with_clusters.columns if col != "cluster_id"]

# Build aggregation dict
agg_dict = {col: get_agg_function(col) for col in agg_cols}

agg_dict_readable = {}
for col, func in agg_dict.items():
    func_name = func.__name__ if hasattr(func, '__name__') else str(func)
    agg_dict_readable[col] = func_name

# Perform the deduplication with groupby and agg
print(f"Original rows: {len(df_with_clusters)}")

# df_with_clusters = df_with_clusters.astype("string")
df_deduped = df_with_clusters.groupby("cluster_id", dropna=False).agg(agg_dict).reset_index()

print(f"Deduplicated rows: {len(df_deduped)}")
print(f"Duplicates removed: {len(df_with_clusters) - len(df_deduped)}")

# df = df.sort_values(by=cols_to_sort_drop).drop(columns=cols_to_sort_drop)
# df_deduped = df_deduped.sort_values(by=cols_to_sort_drop).drop(columns=cols_to_sort_drop)


Original rows: 1070803
Deduplicated rows: 1042771
Duplicates removed: 28032


In [9]:
df_with_clusters

,cluster_id,source_file,record_id,first_name,last_name,job_title,email,work_phone,company_linkedin,facebook,...,primary_n_of_4,general_2024,primary_2024,general_2022,primary_2022,general_2020,primary_2020,home_zipfour,forename_surname_concat_col_name,company_country
638381,ASF227756,[STATE] raw_data_states/2025-12-20\south carol...,ASF227756,Christian,Silver,Owner,<NA>,(843) 341-2454,<NA>,<NA>,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,christian silver,<NA>
638380,ASF146427,[STATE] raw_data_states/2025-12-20\new hampshi...,ASF146427,Chris,Gagnon,Owner,<NA>,(603) 293-2115,<NA>,<NA>,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,chris gagnon,<NA>
638379,ASF164970,[STATE] raw_data_states/2025-12-20\oregon.xlsx,ASF164970,Jeff,Martin,Owner,<NA>,(503) 298-4139,<NA>,https://www.facebook.com/silver-salmon-grille-...,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,jeff martin,<NA>
638378,ASF2293,[STATE] raw_data_states/2025-12-20\alaska.xlsx,ASF2293,David,Coray,Owner,<NA>,(907) 262-4839,<NA>,<NA>,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,david coray,<NA>
638377,ASF120824,[STATE] raw_data_states/2025-12-20\missouri.xlsx,ASF120824,Doris,Kocina,Owner,<NA>,(573) 392-2312,<NA>,<NA>,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,doris kocina,<NA>
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
17,624069,['Zoominfo scraping 3 - Virginia.csv'],624069,Judy,Hample,President/Chief Executive Officer,hample@umw.edu,+1 850-418-1682,<NA>,<NA>,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,judy hample,<NA>
16,624067,['Zoominfo scraping 3 - Virginia.csv'],624067,Dave,Faggert,President,dfaggert@fflaw.com,+1 757-548-3515,<NA>,<NA>,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,dave faggert,<NA>
1070801,624024,['Zoominfo scraping 3 - Virginia.csv'],624024,Mark,Black,President,blackm@fuma.org,+1 434-924-0972,<NA>,https://facebook.com/forkunionmilitary,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,mark black,<NA>
1070802,624025,"['Zoominfo scraping 3 - Virginia.csv', '_Rob M...",624025,Pouyan,Torabi,Founder,pouyan@radiojavan.com,+1 703-981-9054,<NA>,https://facebook.com/radiojavan,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,pouyan torabi,<NA>


In [10]:
df_deduped

,cluster_id,source_file,record_id,first_name,last_name,job_title,email,work_phone,company_linkedin,facebook,...,primary_n_of_4,general_2024,primary_2024,general_2022,primary_2022,general_2020,primary_2020,home_zipfour,forename_surname_concat_col_name,company_country
0,1,[['3rd Muhammad.csv']],[1],Terrel,Davis,Chief Executive Officers,terrel.davis@ttimemanagementllc.com,+1 610-239-8100,<NA>,https://facebook.com/anexinet,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,terrel davis,<NA>
1,10,[['3rd Muhammad.csv']],[10],Jeremy,Smithson,Chief Executive Officer,jeremy@stitchyfish.com,+1 251-929-4477,<NA>,https://www.facebook.com/shopstitchyfish/,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,jeremy smithson,<NA>
2,100,[['[STATE] raw_data_states\\Sasha Files\\Alaba...,[100],Kevin,Weber,President,kevin@oxfoundations.com,(205) 690-7272,<NA>,http://www.facebook.com/a-1-foundation-solutio...,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,kevin weber,<NA>
3,1000,[['Rob Volmer Project master File - 157k RX AD...,[1000],Bryce,Wood,President,bryce@tcboiler.com,+1 844-827-2697,http://www.linkedin.com/company/tc-boiler-piping,https://www.facebook.com/p/tc-boiler-piping-10...,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,bryce wood,<NA>
4,10000,[['3rd Muhammad.csv']],[10000],Shani,Dowell,Founder & Chief Executive Officer,shani@possipit.com,+1 713-659-9549,<NA>,http://facebook.com/possipit,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,shani dowell,<NA>
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1042766,ASF99995,[[STATE] raw_data_states/2025-12-20\minnesota....,[ASF99995],Gene,Messing,Owner,<NA>,(612) 331-2060,<NA>,<NA>,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,gene messing,<NA>
1042767,ASF99996,[[STATE] raw_data_states/2025-12-20\minnesota....,[ASF99996],Chris,Law,Owner,<NA>,(612) 584-4922,<NA>,<NA>,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,chris law,<NA>
1042768,ASF99997,[[STATE] raw_data_states/2025-12-20\minnesota....,[ASF99997],Gavin,Kaysen,Owner,<NA>,(612) 224-9850,<NA>,https://www.facebook.com/spoonandstable,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,gavin kaysen,<NA>
1042769,ASF99998,[[STATE] raw_data_states/2025-12-20\minnesota....,[ASF99998],Rick,Schmitz,Owner,<NA>,(612) 824-3509,<NA>,https://facebook.com/sportsstarphoto/,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,rick schmitz,<NA>


In [11]:

df_deduped.to_csv("csv_outputs/additional_states/deduped_data_owner_president_ceo_founder.csv", index=False)
